# Ceci n'est pas une orbitale
## Seeing how atomic orbitals combine to make molecular orbitals

This notebook follows the approach used in **Chapter 1: Introduction to Structure and Models of Bonding** in *Modern Physical Organic Chemistry*.

The chapter develops molecular orbitals by asking a few qualitative questions:

1. Do the starting orbitals overlap?
2. Are their phases matched or mismatched?
3. Are the starting orbitals close in energy?
4. Does one atom have lower-energy atomic orbitals than the other?
5. How do those choices change the bonding and antibonding molecular orbitals?

The main chapter landmarks are **Table 1.7**, **Figure 1.11**, the ethylene discussion in **Figures 1.14–1.15**, and the formaldehyde discussion in **Figures 1.16–1.17**.

No linear algebra or differential equations are required. The computer performs the calculations. Your job is to connect each control to a chemical idea and explain what changes in the orbital picture.

## How to use the notebook

Run each code cell with the play button at its left. For interactive plots, move **one control at a time**.

For every experiment:

1. Predict what will happen.
2. Change one value.
3. Describe what changed in the picture.
4. Connect the change to the equation or orbital-mixing rule.

Throughout the notebook, two colors represent opposite **signs**, also called opposite **phases**, of an orbital. They do **not** mean positive and negative electrical charge.

# 1. The one equation we need

We will build a molecular orbital from two starting atomic orbitals:

$$
\psi = c_A\phi_A + c_B\phi_B
$$

Read this equation as:

- $\phi_A$ is an atomic orbital on atom A.
- $\phi_B$ is an atomic orbital on atom B.
- $c_A$ and $c_B$ are **mixing numbers**. Their sizes tell us how much each atomic orbital contributes.
- The signs of $c_A$ and $c_B$ tell us whether the two contributions have the same or opposite phase.
- $\psi$ is the resulting molecular orbital.

The equation is an addition problem. In some regions, the two starting orbitals reinforce each other. In other regions, they cancel.

## Set up the interactive plots

Run this cell once. It loads the plotting tools and defines the simple orbital models used in Sections 2–7.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from ipywidgets import FloatSlider, Dropdown, ToggleButtons, HBox, VBox, interactive_output
from IPython.display import display

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def make_grid(limit=5.0, points=280):
    axis = np.linspace(-limit, limit, points)
    X, Y = np.meshgrid(axis, axis)
    return X, Y


def p_orbital(X, Y, center_x=0.0, alpha=0.8, direction="y"):
    """A simple p-shaped function for teaching. It is not a full atomic calculation."""
    x = X - center_x
    radial = np.exp(-alpha * (x*x + Y*Y))
    if direction == "x":
        return x * radial
    return Y * radial


def s_orbital(X, Y, center_x=0.0, alpha=0.8):
    x = X - center_x
    return np.exp(-alpha * (x*x + Y*Y))


def normalize_for_display(field):
    maximum = np.max(np.abs(field))
    if maximum < 1e-12:
        return field
    return field / maximum


def draw_phase_map(ax, X, Y, field, title, centers=None):
    """Continuous phase map: warm and cool colors represent opposite signs."""
    scaled = normalize_for_display(field)
    levels = np.linspace(-1.0, 1.0, 17)
    ax.contourf(X, Y, scaled, levels=levels, cmap="RdBu_r", extend="both")
    ax.contour(X, Y, scaled, levels=[0.0], colors="black", linewidths=1.0)
    if centers:
        for x0, label in centers:
            ax.scatter([x0], [0], s=42, c="black", zorder=5)
            ax.text(x0, -0.35, label, ha="center", va="top", fontsize=10)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlim(X.min(), X.max())
    ax.set_ylim(Y.min(), Y.max())
    ax.set_xticks([])
    ax.set_yticks([])


def draw_isovalue_map(ax, X, Y, field, fraction, title, centers=None):
    """Draw only regions where |orbital| is above a chosen display threshold."""
    scaled = normalize_for_display(field)
    pos = np.ma.masked_where(scaled < fraction, scaled)
    neg = np.ma.masked_where(scaled > -fraction, scaled)
    ax.contourf(X, Y, pos, levels=[fraction, 1.01], colors=["royalblue"], alpha=0.72)
    ax.contourf(X, Y, neg, levels=[-1.01, -fraction], colors=["tomato"], alpha=0.72)
    ax.contour(X, Y, scaled, levels=[0.0], colors="black", linewidths=0.9)
    if centers:
        for x0, label in centers:
            ax.scatter([x0], [0], s=42, c="black", zorder=5)
            ax.text(x0, -0.35, label, ha="center", va="top", fontsize=10)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlim(X.min(), X.max())
    ax.set_ylim(Y.min(), Y.max())
    ax.set_xticks([])
    ax.set_yticks([])


def orient_coefficients(vector):
    """Choose one of the two equivalent global signs for a tidy display."""
    vector = np.asarray(vector, dtype=float).copy()
    largest = np.argmax(np.abs(vector))
    if vector[largest] < 0:
        vector *= -1
    return vector

print("Setup complete.")

# 2. Add two p orbitals

For two neighboring p orbitals, the same equation applies:

$$
\psi = c_A\phi_A + c_B\phi_B
$$

Use the sliders to change $c_A$ and $c_B$.

- Equal signs usually give an **in-phase** combination for the side-by-side p orbitals shown here.
- Opposite signs give an **out-of-phase** combination.
- A larger coefficient gives that atom a larger contribution to the molecular orbital.

The black line is a node, where $\psi=0$.

In [ ]:
def plot_two_p_orbitals(c_A=1.0, c_B=1.0, distance=2.4, compactness=0.8):
    X, Y = make_grid(limit=4.6, points=260)
    left = -distance / 2.0
    right = distance / 2.0

    phi_A = p_orbital(X, Y, center_x=left, alpha=compactness, direction="y")
    phi_B = p_orbital(X, Y, center_x=right, alpha=compactness, direction="y")
    psi = c_A * phi_A + c_B * phi_B

    if np.max(np.abs(psi)) < 1e-12:
        print("Both coefficients are zero, so the trial molecular orbital is zero everywhere.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    centers = [(left, "A"), (right, "B")]
    draw_phase_map(axes[0], X, Y, phi_A, r"starting orbital $\phi_A$", centers)
    draw_phase_map(axes[1], X, Y, phi_B, r"starting orbital $\phi_B$", centers)
    draw_phase_map(axes[2], X, Y, psi, r"resulting MO $\psi$", centers)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    relation = "same phase" if c_A * c_B > 0 else "opposite phase" if c_A * c_B < 0 else "only one orbital contributes"
    print(f"Current equation: psi = ({c_A:+.2f}) phi_A + ({c_B:+.2f}) phi_B")
    print(f"Phase relationship: {relation}")
    if abs(c_A) > abs(c_B):
        print("Atom A has the larger coefficient magnitude.")
    elif abs(c_B) > abs(c_A):
        print("Atom B has the larger coefficient magnitude.")
    else:
        print("The two coefficient magnitudes are equal.")


controls_2 = {
    "c_A": FloatSlider(value=1.0, min=-1.5, max=1.5, step=0.1,
                       description="c_A", continuous_update=False),
    "c_B": FloatSlider(value=1.0, min=-1.5, max=1.5, step=0.1,
                       description="c_B", continuous_update=False),
    "distance": FloatSlider(value=2.4, min=1.4, max=4.0, step=0.1,
                            description="distance", continuous_update=False),
    "compactness": FloatSlider(value=0.8, min=0.4, max=1.6, step=0.1,
                               description="compact", continuous_update=False),
}

output_2 = interactive_output(plot_two_p_orbitals, controls_2)
display(VBox([
    HBox([controls_2["c_A"], controls_2["c_B"]]),
    HBox([controls_2["distance"], controls_2["compactness"]]),
]), output_2)

### Questions for Section 2

1. Set $c_A=1$ and $c_B=1$. Is there a node between the atoms?
2. Set $c_A=1$ and $c_B=-1$. What new node appears?
3. Set $c_A=1$ and $c_B=0.4$. Which atom contributes more strongly?
4. Change both coefficients from positive to negative. The colors reverse. Did the nodal pattern change?
5. Complete this sentence: “Changing the sign of one coefficient changes ________, while changing the size of one coefficient changes ________.”

# 3. The orbital and the electron density are not the same thing

The orbital $\psi$ can be positive or negative. Its sign carries phase information.

A simple orbital-density picture is obtained by squaring the orbital:

$$
\rho = \psi^2
$$

Squaring removes the sign. This is why two orbitals that differ only by an overall sign have the same density.

Use the control below to reverse the global sign. Watch what happens to $\psi$ and to $\psi^2$.

In [ ]:
def plot_wave_and_density(combination="bonding", global_sign="+1"):
    x = np.linspace(-6.0, 6.0, 1400)
    distance = 2.4
    alpha = 0.75
    phi_A = np.exp(-alpha * (x + distance/2.0)**2)
    phi_B = np.exp(-alpha * (x - distance/2.0)**2)

    relative_sign = 1.0 if combination == "bonding" else -1.0
    sign = 1.0 if global_sign == "+1" else -1.0
    psi = sign * (phi_A + relative_sign * phi_B)
    psi = psi / np.sqrt(np.trapz(psi*psi, x))
    density = psi * psi

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(x, psi, linewidth=2.2)
    axes[0].axhline(0, linewidth=0.8, color="black")
    axes[0].axvline(-distance/2.0, linestyle=":", color="black")
    axes[0].axvline(distance/2.0, linestyle=":", color="black")
    axes[0].set_title(r"orbital amplitude $\psi$")
    axes[0].set_xlabel("position")
    axes[0].set_ylabel("amplitude")

    axes[1].plot(x, density, linewidth=2.2)
    axes[1].axvline(-distance/2.0, linestyle=":", color="black")
    axes[1].axvline(distance/2.0, linestyle=":", color="black")
    axes[1].set_title(r"orbital density $\psi^2$")
    axes[1].set_xlabel("position")
    axes[1].set_ylabel("density")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


controls_3 = {
    "combination": Dropdown(options=["bonding", "antibonding"], value="bonding",
                            description="orbital"),
    "global_sign": ToggleButtons(options=["+1", "-1"], value="+1",
                                 description="global sign"),
}
output_3 = interactive_output(plot_wave_and_density, controls_3)
display(HBox([controls_3["combination"], controls_3["global_sign"]]), output_3)

### Questions for Section 3

1. Reverse the global sign. What changes in the $\psi$ plot?
2. What changes in the $\psi^2$ plot?
3. Why do the two colors in an orbital plot not represent positive and negative charge?
4. Which plot keeps the information needed to identify a node caused by a phase change?

# 4. Overlap controls how strongly orbitals interact

Chapter 1 states that greater overlap produces a stronger orbital interaction.

A formal overlap measure is written as

$$
S = \int \phi_A\phi_B\,d\tau
$$

You do not need to evaluate this integral by hand. It simply measures how much the two orbitals occupy the same region of space with matching or mismatching phase.

In the cartoon model below:

- moving the atoms closer increases overlap;
- making the orbitals more compact usually decreases overlap at the same distance;
- stronger overlap produces a larger separation between the bonding and antibonding energy levels.

The energy values are teaching-model units, not experimental numbers.

In [ ]:
def plot_overlap(distance=2.4, compactness=0.8):
    X, Y = make_grid(limit=5.0, points=280)
    dx = X[0, 1] - X[0, 0]
    dy = Y[1, 0] - Y[0, 0]
    left = -distance/2.0
    right = distance/2.0

    phi_A = s_orbital(X, Y, center_x=left, alpha=compactness)
    phi_B = s_orbital(X, Y, center_x=right, alpha=compactness)

    phi_A = phi_A / np.sqrt(np.sum(phi_A*phi_A) * dx * dy)
    phi_B = phi_B / np.sqrt(np.sum(phi_B*phi_B) * dx * dy)
    overlap = float(np.sum(phi_A * phi_B) * dx * dy)

    splitting = 1.6 * abs(overlap)
    bonding_E = -splitting / 2.0
    antibonding_E = splitting / 2.0

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1))
    centers = [(left, "A"), (right, "B")]
    draw_phase_map(axes[0], X, Y, phi_A + phi_B,
                   "in-phase combination", centers)

    ax = axes[1]
    ax.hlines(0, 0.05, 0.30, linewidth=2.2, color="black")
    ax.hlines(0, 0.70, 0.95, linewidth=2.2, color="black")
    ax.hlines(bonding_E, 0.38, 0.62, linewidth=3.0, color="black")
    ax.hlines(antibonding_E, 0.38, 0.62, linewidth=3.0, color="black")
    ax.plot([0.30, 0.38], [0, bonding_E], linestyle=":", color="gray")
    ax.plot([0.30, 0.38], [0, antibonding_E], linestyle=":", color="gray")
    ax.plot([0.70, 0.62], [0, bonding_E], linestyle=":", color="gray")
    ax.plot([0.70, 0.62], [0, antibonding_E], linestyle=":", color="gray")
    ax.text(0.175, 0.04, "A", ha="center")
    ax.text(0.825, 0.04, "B", ha="center")
    ax.text(0.50, bonding_E-0.06, "bonding", ha="center", va="top")
    ax.text(0.50, antibonding_E+0.06, "antibonding", ha="center", va="bottom")
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.9, 0.9)
    ax.set_xticks([])
    ax.set_ylabel("relative energy")
    ax.set_title("more overlap → larger energy splitting")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    print(f"Overlap score S = {overlap:.3f}")
    print(f"Energy splitting in this teaching model = {splitting:.3f}")


controls_4 = {
    "distance": FloatSlider(value=2.4, min=1.2, max=4.5, step=0.1,
                            description="distance", continuous_update=False),
    "compactness": FloatSlider(value=0.8, min=0.35, max=1.8, step=0.05,
                               description="compact", continuous_update=False),
}
output_4 = interactive_output(plot_overlap, controls_4)
display(HBox([controls_4["distance"], controls_4["compactness"]]), output_4)

### Questions for Section 4

1. Increase the distance. What happens to overlap?
2. What happens to the bonding–antibonding energy separation?
3. At a fixed distance, make the orbitals more compact. What happens?
4. Use the chapter rule to complete the statement: “A geometry change matters when it changes orbital ________.”

# 5. Starting orbital energies control mixing and polarization

Figure 1.11 compares two cases:

- **equal-energy starting orbitals**, which mix strongly and contribute about equally;
- **unequal-energy starting orbitals**, which mix less and produce polarized molecular orbitals.

The lower molecular orbital contains more of the lower-energy starting orbital. The higher molecular orbital contains more of the higher-energy starting orbital.

The computer will find the two combinations. You only need to read the resulting coefficients in

$$
\psi = c_A\phi_A + c_B\phi_B.
$$

Move the **B lower by** slider. A value of zero is ethylene-like because the two carbon p orbitals begin at the same energy. A larger value is carbonyl-like because an oxygen p orbital lies lower in energy than a carbon p orbital.

In [ ]:
def plot_energy_mismatch(B_lower_by=0.0, mixing_strength=0.30):
    # A simple two-orbital mixing model. The matrix calculation stays behind the scenes.
    E_A = 0.0
    E_B = -float(B_lower_by)
    interaction = -float(mixing_strength)
    model = np.array([[E_A, interaction], [interaction, E_B]], dtype=float)
    energies, vectors = np.linalg.eigh(model)

    lower_coeff = orient_coefficients(vectors[:, 0])
    upper_coeff = orient_coefficients(vectors[:, 1])

    X, Y = make_grid(limit=4.6, points=260)
    distance = 2.4
    left = -distance/2.0
    right = distance/2.0
    phi_A = p_orbital(X, Y, left, alpha=0.8, direction="y")
    phi_B = p_orbital(X, Y, right, alpha=0.8, direction="y")
    lower_field = lower_coeff[0]*phi_A + lower_coeff[1]*phi_B
    upper_field = upper_coeff[0]*phi_A + upper_coeff[1]*phi_B

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
    centers = [(left, "A"), (right, "B")]
    draw_phase_map(
        axes[0], X, Y, lower_field,
        f"lower MO\n{lower_coeff[0]:+.2f} A {lower_coeff[1]:+.2f} B", centers
    )
    draw_phase_map(
        axes[1], X, Y, upper_field,
        f"upper MO\n{upper_coeff[0]:+.2f} A {upper_coeff[1]:+.2f} B", centers
    )

    ax = axes[2]
    ax.hlines(E_A, 0.05, 0.28, linewidth=2.2, color="black")
    ax.hlines(E_B, 0.72, 0.95, linewidth=2.2, color="black")
    ax.hlines(energies[0], 0.38, 0.62, linewidth=3.0, color="black")
    ax.hlines(energies[1], 0.38, 0.62, linewidth=3.0, color="black")
    ax.text(0.165, E_A+0.05, "A", ha="center")
    ax.text(0.835, E_B+0.05, "B", ha="center")
    ax.text(0.50, energies[0]-0.06, "lower MO", ha="center", va="top")
    ax.text(0.50, energies[1]+0.06, "upper MO", ha="center", va="bottom")
    ax.set_xlim(0, 1)
    ymin = min(E_B, energies[0]) - 0.35
    ymax = max(E_A, energies[1]) + 0.35
    ax.set_ylim(ymin, ymax)
    ax.set_xticks([])
    ax.set_ylabel("relative energy")
    ax.set_title("starting and final energy levels")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    lower_A = 100 * lower_coeff[0]**2
    lower_B = 100 * lower_coeff[1]**2
    upper_A = 100 * upper_coeff[0]**2
    upper_B = 100 * upper_coeff[1]**2
    print(f"Lower MO: about {lower_A:.1f}% A and {lower_B:.1f}% B in this model")
    print(f"Upper MO: about {upper_A:.1f}% A and {upper_B:.1f}% B in this model")
    print("The percentages are coefficient-squared values from a simplified teaching model.")


controls_5 = {
    "B_lower_by": FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1,
                              description="B lower by", continuous_update=False),
    "mixing_strength": FloatSlider(value=0.30, min=0.05, max=0.60, step=0.05,
                                   description="mixing", continuous_update=False),
}
output_5 = interactive_output(plot_energy_mismatch, controls_5)
display(HBox([controls_5["B_lower_by"], controls_5["mixing_strength"]]), output_5)

### Questions for Section 5

1. Set **B lower by = 0**. How do the two coefficient magnitudes compare?
2. Increase **B lower by**. Which starting orbital dominates the lower MO?
3. Which starting orbital dominates the upper MO?
4. At a large energy difference, increase the mixing strength. Do the orbitals become more or less polarized?
5. State the chapter rule in your own words: orbitals that are closer in energy mix ________.

# 6. Ethylene: the $\pi$ and $\pi^*$ orbitals

Chapter 1 constructs ethylene by combining two carbon p orbitals. Because the two atoms are the same, the starting p orbitals have equal energy.

The two combinations are

$$
\pi \propto p_A + p_B
$$

and

$$
\pi^* \propto p_A - p_B.
$$

- The $\pi$ orbital is bonding and lower in energy.
- The $\pi^*$ orbital is antibonding and higher in energy.
- Ethylene has two $\pi$ electrons, so $\pi$ is occupied and is the HOMO.
- $\pi^*$ is empty and is the LUMO.

Use the display threshold to change the apparent lobe size. This does not change either molecular orbital.

In [ ]:
def plot_ethylene(distance=2.4, display_threshold=0.20, global_phase="normal"):
    X, Y = make_grid(limit=4.6, points=280)
    left = -distance/2.0
    right = distance/2.0
    p_A = p_orbital(X, Y, left, alpha=0.8, direction="y")
    p_B = p_orbital(X, Y, right, alpha=0.8, direction="y")
    phase = 1.0 if global_phase == "normal" else -1.0
    pi = phase * (p_A + p_B)
    pi_star = phase * (p_A - p_B)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
    centers = [(left, "C"), (right, "C")]
    draw_isovalue_map(axes[0], X, Y, pi, display_threshold,
                      r"$\pi$ bonding MO (HOMO)", centers)
    draw_isovalue_map(axes[1], X, Y, pi_star, display_threshold,
                      r"$\pi^*$ antibonding MO (LUMO)", centers)

    ax = axes[2]
    ax.hlines(-0.6, 0.25, 0.75, linewidth=3.0, color="black")
    ax.hlines(0.6, 0.25, 0.75, linewidth=3.0, color="black")
    ax.text(0.50, -0.52, r"$\pi$   ↑↓", ha="center", va="bottom", fontsize=13)
    ax.text(0.50, 0.68, r"$\pi^*$   empty", ha="center", va="bottom", fontsize=13)
    ax.text(0.50, -0.88, "HOMO", ha="center", fontsize=11)
    ax.text(0.50, 0.92, "LUMO", ha="center", fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("ethylene π energy levels")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    print(f"Display threshold = {display_threshold:.2f} of the maximum orbital amplitude")
    print("Changing the threshold changes the picture, not the calculated pi or pi* function.")


controls_6 = {
    "distance": FloatSlider(value=2.4, min=1.5, max=3.8, step=0.1,
                            description="C-C distance", continuous_update=False),
    "display_threshold": FloatSlider(value=0.20, min=0.05, max=0.60, step=0.05,
                                     description="threshold", continuous_update=False),
    "global_phase": ToggleButtons(options=["normal", "reversed"], value="normal",
                                  description="global phase"),
}
output_6 = interactive_output(plot_ethylene, controls_6)
display(VBox([
    HBox([controls_6["distance"], controls_6["display_threshold"]]),
    controls_6["global_phase"],
]), output_6)

### Questions for Section 6

1. Which orbital has a node between the two carbons?
2. Why is that orbital antibonding?
3. Why is $\pi$ the HOMO of ethylene?
4. Why is $\pi^*$ the LUMO?
5. Increase the display threshold. Did the node move? Did the orbital energy change?
6. Reverse the global phase. What changed chemically?

# 7. Formaldehyde: unequal orbital energies produce polarization

Formaldehyde and ethylene have the same number of valence electrons, but oxygen changes the orbital-energy pattern.

Chapter 1 emphasizes three points:

1. Oxygen is more electronegative than carbon.
2. Oxygen atomic orbitals therefore begin lower in energy.
3. The bonding $\pi$ MO is polarized toward oxygen, while the antibonding $\pi^*$ MO is polarized toward carbon.

The same LCAO equation still applies:

$$
\psi = c_C p_C + c_O p_O.
$$

The coefficients are no longer equal.

**Important:** In formaldehyde, $\pi^*$ is the LUMO, but the HOMO is an oxygen-rich lone-pair molecular orbital. This section isolates the $\pi/\pi^*$ pair so that we can see how the carbon–oxygen energy difference polarizes them.

In [ ]:
def plot_formaldehyde(oxygen_lowering=1.0, mixing_strength=0.32,
                      oxygen_compactness=1.25, display_threshold=0.18):
    E_C = 0.0
    E_O = -float(oxygen_lowering)
    interaction = -float(mixing_strength)
    model = np.array([[E_C, interaction], [interaction, E_O]], dtype=float)
    energies, vectors = np.linalg.eigh(model)
    pi_coeff = orient_coefficients(vectors[:, 0])
    pi_star_coeff = orient_coefficients(vectors[:, 1])

    X, Y = make_grid(limit=4.8, points=290)
    distance = 2.3
    x_C = -distance/2.0
    x_O = distance/2.0
    p_C = p_orbital(X, Y, x_C, alpha=0.75, direction="y")
    p_O = p_orbital(X, Y, x_O, alpha=oxygen_compactness, direction="y")
    pi_field = pi_coeff[0]*p_C + pi_coeff[1]*p_O
    pi_star_field = pi_star_coeff[0]*p_C + pi_star_coeff[1]*p_O

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
    centers = [(x_C, "C"), (x_O, "O")]
    draw_isovalue_map(
        axes[0], X, Y, pi_field, display_threshold,
        f"bonding $\pi$ MO\n{pi_coeff[0]:+.2f} C {pi_coeff[1]:+.2f} O", centers
    )
    draw_isovalue_map(
        axes[1], X, Y, pi_star_field, display_threshold,
        f"antibonding $\pi^*$ MO (LUMO)\n{pi_star_coeff[0]:+.2f} C {pi_star_coeff[1]:+.2f} O", centers
    )

    ax = axes[2]
    ax.hlines(E_C, 0.05, 0.28, linewidth=2.2, color="black")
    ax.hlines(E_O, 0.72, 0.95, linewidth=2.2, color="black")
    ax.hlines(energies[0], 0.38, 0.62, linewidth=3.0, color="black")
    ax.hlines(energies[1], 0.38, 0.62, linewidth=3.0, color="black")
    ax.text(0.165, E_C+0.05, "C p", ha="center")
    ax.text(0.835, E_O+0.05, "O p", ha="center")
    ax.text(0.50, energies[0]-0.06, r"$\pi$", ha="center", va="top", fontsize=13)
    ax.text(0.50, energies[1]+0.06, r"$\pi^*$", ha="center", va="bottom", fontsize=13)
    ax.set_xlim(0, 1)
    ax.set_ylim(min(E_O, energies[0])-0.35, max(E_C, energies[1])+0.35)
    ax.set_xticks([])
    ax.set_ylabel("relative energy")
    ax.set_title("carbon and oxygen start at different energies")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    pi_C = 100*pi_coeff[0]**2
    pi_O = 100*pi_coeff[1]**2
    pistar_C = 100*pi_star_coeff[0]**2
    pistar_O = 100*pi_star_coeff[1]**2
    print(f"Bonding pi model: {pi_C:.1f}% carbon and {pi_O:.1f}% oxygen")
    print(f"Antibonding pi* model: {pistar_C:.1f}% carbon and {pistar_O:.1f}% oxygen")
    print("The larger carbon coefficient in pi* helps explain why nucleophiles attack carbonyl carbon.")


controls_7 = {
    "oxygen_lowering": FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1,
                                   description="O lower by", continuous_update=False),
    "mixing_strength": FloatSlider(value=0.32, min=0.05, max=0.60, step=0.05,
                                   description="mixing", continuous_update=False),
    "oxygen_compactness": FloatSlider(value=1.25, min=0.70, max=2.00, step=0.05,
                                      description="O compact", continuous_update=False),
    "display_threshold": FloatSlider(value=0.18, min=0.05, max=0.55, step=0.05,
                                     description="threshold", continuous_update=False),
}
output_7 = interactive_output(plot_formaldehyde, controls_7)
display(VBox([
    HBox([controls_7["oxygen_lowering"], controls_7["mixing_strength"]]),
    HBox([controls_7["oxygen_compactness"], controls_7["display_threshold"]]),
]), output_7)

### Questions for Section 7

1. Set **O lower by = 0**. How does the result resemble ethylene?
2. Lower the oxygen p orbital. Which atom contributes more to the bonding $\pi$ MO?
3. Which atom contributes more to the $\pi^*$ MO?
4. Why is the carbonyl $\pi^*$ orbital useful for predicting nucleophilic attack?
5. Why should you not identify the bonding $\pi$ orbital as the HOMO of formaldehyde?
6. The chapter warns that oxygen orbitals are smaller. Change **O compact** and explain why apparent lobe size is not identical to coefficient size.

# 8. “This is not an orbital”: changing the display threshold

Most three-dimensional orbital images show a surface at a selected orbital value. The software draws locations where

$$
\psi = +\tau
\qquad\text{or}\qquad
\psi = -\tau,
$$

where $\tau$ is the chosen threshold, often called the **isovalue**.

Changing $\tau$ changes how much of the function is displayed. It does not change the function, its nodes, its coefficients, or its energy.

In [ ]:
def plot_threshold_demo(threshold=0.15):
    X, Y = make_grid(limit=4.6, points=290)
    distance = 2.4
    left = -distance/2.0
    right = distance/2.0
    p_A = p_orbital(X, Y, left, alpha=0.8, direction="y")
    p_B = p_orbital(X, Y, right, alpha=0.8, direction="y")
    fixed_orbital = p_A - p_B

    fig, ax = plt.subplots(figsize=(6.0, 4.8))
    draw_isovalue_map(ax, X, Y, fixed_orbital, threshold,
                      rf"same $\pi^*$ orbital, threshold = {threshold:.2f}",
                      [(left, "C"), (right, "C")])
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    print("The underlying orbital array was not recalculated. Only the display cutoff changed.")


threshold_control = FloatSlider(value=0.15, min=0.03, max=0.70, step=0.02,
                                description="threshold", continuous_update=False)
threshold_output = interactive_output(plot_threshold_demo, {"threshold": threshold_control})
display(threshold_control, threshold_output)

### Questions for Section 8

1. Raise the threshold. Why do the lobes appear smaller?
2. Does the node between the carbons move?
3. Does the equation for the orbital change?
4. Explain the title “Ceci n'est pas une orbitale” in one or two sentences.

# 9. Optional: calculate and display an orbital from a SMILES string

The earlier sections used simple models so that each chemical idea was easy to see. This section runs a small Hartree–Fock calculation and displays a calculated molecular orbital.

The workflow is

$$
\text{SMILES} \rightarrow \text{3D geometry} \rightarrow \text{MO calculation} \rightarrow \text{orbital grid} \rightarrow \text{displayed surface}.
$$

Keep the first tests small. Recommended examples are:

| Molecule | SMILES |
|---|---|
| ethylene | `C=C` |
| formaldehyde | `C=O` |
| butadiene | `C=CC=C` |
| benzene | `c1ccccc1` |
| pyridine | `n1ccccc1` |

This beginner version is limited to neutral or charged **closed-shell, even-electron** molecules. The generated geometry is adequate for a teaching visualization but is not a publication-quality quantum-optimized structure.

In [ ]:
# @title Install the chemistry packages once per Colab session
!pip -q install rdkit pyscf py3Dmol
print("RDKit, PySCF, and py3Dmol are ready.")

In [ ]:
# @title Choose a small molecule and an orbital
SMILES = "C=C"  # @param {type:"string"}
CHARGE = 0  # @param {type:"integer"}
BASIS = "sto-3g"  # @param ["sto-3g", "3-21g", "6-31g"]
ORBITAL = "HOMO"  # @param ["HOMO-2", "HOMO-1", "HOMO", "LUMO", "LUMO+1", "LUMO+2"]
GRID = 40  # @param {type:"slider", min:30, max:55, step:5}
ISOVALUE = 0.04  # @param {type:"slider", min:0.01, max:0.10, step:0.005}

print("Settings recorded. Run the next two cells.")

In [ ]:
import re
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto, scf, lib
from pyscf.tools import cubegen
import py3Dmol

lib.num_threads(2)


def make_3d_geometry(smiles, max_total_atoms=35, seed=2026):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError("The SMILES string could not be read.")

    molecule = Chem.AddHs(molecule)
    if molecule.GetNumAtoms() > max_total_atoms:
        raise ValueError(
            f"This teaching notebook allows at most {max_total_atoms} atoms including hydrogen."
        )

    settings = AllChem.ETKDGv3()
    settings.randomSeed = int(seed)
    status = AllChem.EmbedMolecule(molecule, settings)
    if status != 0:
        settings.useRandomCoords = True
        status = AllChem.EmbedMolecule(molecule, settings)
    if status != 0:
        raise RuntimeError("A 3D structure could not be generated for this molecule.")

    if AllChem.MMFFHasAllMoleculeParams(molecule):
        AllChem.MMFFOptimizeMolecule(molecule, mmffVariant="MMFF94s", maxIters=400)
        geometry_note = "ETKDG + MMFF94s"
    else:
        AllChem.UFFOptimizeMolecule(molecule, maxIters=400)
        geometry_note = "ETKDG + UFF"

    conformer = molecule.GetConformer()
    lines = []
    for atom in molecule.GetAtoms():
        position = conformer.GetAtomPosition(atom.GetIdx())
        lines.append(
            f"{atom.GetSymbol()} {position.x:.9f} {position.y:.9f} {position.z:.9f}"
        )

    atom_block = "\n".join(lines)
    xyz_block = f"{molecule.GetNumAtoms()}\nGenerated from {smiles}\n{atom_block}\n"
    return molecule, atom_block, xyz_block, geometry_note


def select_closed_shell_orbital(label, occupations):
    occupied = np.flatnonzero(occupations > 1e-8)
    virtual = np.flatnonzero(occupations <= 1e-8)
    if len(occupied) == 0:
        raise ValueError("No occupied orbitals were found.")

    homo = int(occupied[-1])
    lumo = int(virtual[0]) if len(virtual) else None

    if label.startswith("HOMO"):
        offsets = {"HOMO-2": -2, "HOMO-1": -1, "HOMO": 0}
        index = homo + offsets[label]
    else:
        if lumo is None:
            raise ValueError("This basis did not produce a virtual orbital to display.")
        offsets = {"LUMO": 0, "LUMO+1": 1, "LUMO+2": 2}
        index = lumo + offsets[label]

    if index < 0 or index >= len(occupations):
        raise IndexError(f"{label} does not exist for this calculation.")
    return index, homo, lumo


def calculate_orbital(smiles, charge=0, basis="sto-3g", orbital_label="HOMO", grid=40):
    rd_mol, atom_block, xyz_block, geometry_note = make_3d_geometry(smiles)
    nuclear_charge = sum(atom.GetAtomicNum() for atom in rd_mol.GetAtoms())
    electron_count = nuclear_charge - int(charge)

    if electron_count % 2 != 0:
        raise ValueError(
            "This beginner notebook uses a closed-shell calculation and therefore requires "
            "an even number of electrons. Choose a different molecule or charge."
        )

    qm_mol = gto.M(
        atom=atom_block,
        basis=basis,
        unit="Angstrom",
        charge=int(charge),
        spin=0,
        verbose=0,
    )

    calculation = scf.RHF(qm_mol)
    calculation.conv_tol = 1e-8
    calculation.max_cycle = 100
    total_energy = calculation.kernel()

    index, homo, lumo = select_closed_shell_orbital(
        orbital_label, np.asarray(calculation.mo_occ)
    )

    safe_smiles = re.sub(r"[^A-Za-z0-9_.+-]+", "_", smiles)
    cube_name = f"{safe_smiles}_{basis}_{orbital_label}.cube"
    cubegen.orbital(
        qm_mol,
        cube_name,
        calculation.mo_coeff[:, index],
        nx=int(grid), ny=int(grid), nz=int(grid),
    )

    return {
        "rd_mol": rd_mol,
        "qm_mol": qm_mol,
        "calculation": calculation,
        "xyz_block": xyz_block,
        "geometry_note": geometry_note,
        "electron_count": electron_count,
        "total_energy": total_energy,
        "index": index,
        "homo": homo,
        "lumo": lumo,
        "cube_name": cube_name,
    }


def display_cube(result, isovalue=0.04, width=820, height=540):
    with open(result["cube_name"], "r") as file:
        cube = file.read()

    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(result["xyz_block"], "xyz")
    viewer.setStyle({"stick": {}, "sphere": {"scale": 0.27}})
    viewer.addVolumetricData(
        cube, "cube",
        {"isoval": float(isovalue), "color": "blue", "opacity": 0.72},
    )
    viewer.addVolumetricData(
        cube, "cube",
        {"isoval": -float(isovalue), "color": "red", "opacity": 0.72},
    )
    viewer.zoomTo()
    return viewer

print("Calculation functions are ready.")

In [ ]:
# @title Run the calculation and show the orbital
result = calculate_orbital(
    smiles=SMILES,
    charge=CHARGE,
    basis=BASIS,
    orbital_label=ORBITAL,
    grid=GRID,
)

print(f"SMILES: {SMILES}")
print(f"Geometry method: {result['geometry_note']}")
print(f"Electron count: {result['electron_count']}")
print(f"SCF converged: {result['calculation'].converged}")
print(f"HOMO index: {result['homo']}")
print(f"LUMO index: {result['lumo']}")
print(f"Displayed orbital: {ORBITAL}, index {result['index']}")
print(f"Orbital energy: {result['calculation'].mo_energy[result['index']]:.6f} Hartree")

viewer = display_cube(result, ISOVALUE)
viewer.show()

print("Blue and red represent opposite orbital phases, not opposite charges.")
print("Change ISOVALUE and rerun only this cell to change the displayed surface.")

### Questions for the calculated-orbital section

1. Calculate ethylene. Does its HOMO resemble the bonding $\pi$ orbital from Section 6?
2. Display ethylene's LUMO. Where is the additional node?
3. Calculate formaldehyde. Remember that its HOMO is expected to be an oxygen-rich lone-pair MO, while its LUMO is $\pi^*$.
4. Change only the isovalue. What changes, and what stays fixed?
5. Change the basis from `sto-3g` to `6-31g`. In the equation $\psi=\sum c\phi$, this changes the set of functions used to describe the atomic orbitals.
6. Explain one limitation introduced by generating the geometry from SMILES.

# 10. LLM-assisted extension

Use a free-tier large language model as a coding partner. The LLM should help you make **one controlled change**, not rewrite the entire notebook.

Choose one extension:

- add a menu for additional small molecules;
- display two orbitals side by side;
- compare ethylene and formaldehyde;
- add a control that changes one bond length;
- add a table of orbital energies;
- add a control for a different orbital number;
- compare two basis sets while keeping the geometry fixed.

Use this prompt as a starting point:

```text
I am a sophomore chemistry student modifying a Google Colab notebook about
molecular orbitals. The key equation is

psi = c_A phi_A + c_B phi_B.

I want to add this one feature: [DESCRIBE ONE FEATURE].

Before giving code, explain in plain chemistry language:
1. which quantity is changing;
2. what I should predict about phase, nodes, coefficients, overlap, energy,
   or only the displayed isosurface;
3. what I should keep fixed for a fair comparison.

Then give the smallest code change possible. Do not assume that I know linear
algebra or differential equations. Tell me how to test whether the code worked.
```

The LLM may produce working code and still give a poor chemical explanation. Check the output against the chapter rules and against a simple limiting case such as equal coefficients, zero energy difference, very large atom separation, or a changed isovalue.

# 11. Experiment record

Complete at least four experiments. Change one item at a time.

| Experiment | What I changed | My prediction | What I observed | Chapter rule or equation used | How I checked the LLM or code |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |
| 4 |  |  |  |  |  |

At least one experiment must document an LLM statement or code suggestion that was incomplete, misleading, or wrong.

# 12. Final questions

Answer in complete sentences.

1. In $\psi=c_A\phi_A+c_B\phi_B$, what do the coefficient size and coefficient sign control?
2. What is the difference between an in-phase and an out-of-phase combination?
3. Why does an antibonding combination contain a node between the atoms?
4. How does moving two atoms farther apart change overlap and orbital mixing?
5. Why do equal-energy carbon p orbitals contribute equally to ethylene's $\pi$ and $\pi^*$ orbitals?
6. Why are the carbonyl $\pi$ and $\pi^*$ orbitals polarized in opposite directions?
7. Why is the $\pi^*$ LUMO of a carbonyl larger on carbon in the qualitative model?
8. Why is the HOMO of formaldehyde not simply its bonding $\pi$ orbital?
9. What does changing an isovalue alter? What does it not alter?
10. Explain why a colored orbital picture is a representation of an orbital rather than the orbital itself.
11. Identify one useful contribution from your LLM and one statement you had to correct or qualify.

# 13. Submission

Submit:

1. the completed Colab notebook;
2. four controlled experiments;
3. one working LLM-assisted extension;
4. the prompt or prompts used;
5. your final-question answers;
6. a short description of one LLM limitation you identified.

A strong submission does not merely contain attractive pictures. It connects each picture to a coefficient, phase relationship, overlap change, starting orbital energy, molecular geometry, or display threshold.

## Source and scope

This notebook is an original teaching activity organized around the qualitative molecular-orbital treatment in Chapter 1 of *Modern Physical Organic Chemistry* by Eric V. Anslyn and Dennis A. Dougherty. It follows the chapter's orbital-mixing rules and its ethylene and formaldehyde examples, but the interactive diagrams and code are newly generated for this activity.

The simple orbital functions and energy models in Sections 2–8 are qualitative teaching models. The optional PySCF section performs a real, low-cost electronic-structure calculation, but it still uses a generated starting geometry and modest computational settings suitable for a free Colab runtime.